In [18]:
import operator
from typing import TypedDict, Annotated, Sequence

from pydantic import BaseModel, Field
from IPython.display import Image, display
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END, START

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings

class LanguageIntent(BaseModel):
    """data extracted from the user's input for translation."""
    source_lang_code: str = Field(description="The language code (e.g., 'hindi') of the original user query")
    target_lang_code: str = Field(description="The language code (e.g., 'marathi') of the desired translation output")
    query_for_retrieval: str = Field(description="The phrase from the user's input that needs translation")

class TranslationState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    source_lang: str           
    target_lang: str           
    query_for_retrieval: str   

    retrieved_context: str     # top search results from ChromaDB

    final_translation: str     

    tool_call_result: str      

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embedding_model = None
vector_store = None


In [20]:
def identify_intent(state: TranslationState) -> dict:
    """
    Uses Pydantic to extract the source and target language codes and translation query from user's message. Returns a dict containing the extracted fields to update the state
    
    arguments: 
        state: current Langgraph state
    """

    user_message=state["messages"][-1].content

    prompt = ChatPromptTemplate.from_messages([
        ("system", 
         "You are a language intent extraction agent. Task: analyze user's request, "
         "identify source language, target language and the text that requires translation. "
         "You must output a JSON object that adheres to the schema. "
         "Use ISO 639 codes (eg Hindi='hi', Tamil='ta'). "
         "Assume  source language is the one used in the prompt if we do not have any explicit source mentioned, "
         "and the target language is the one that's explicitly requested by the user or otherwise implied."
        ),
        ("human", "User Request: {user_input}")
    ])

    extraction_chain = prompt | llm.with_structured_output(LanguageIntent) # | is langchain's pipeline operator, passes prompt to LLM with the LanguageIntent schema
    try:
        intent_data = extraction_chain.invoke({
            "user_input": user_message 
        })
        
        return {
            "source_lang": intent_data.source_lang_code,
            "target_lang": intent_data.target_lang_code,
            "query_for_retrieval": intent_data.query_for_retrieval,
        }
    except Exception as e:
        print(f"Error during intent identification: {e}")
        return {
            "source_lang": "error",
            "target_lang": "error",
            "query_for_retrieval": "Error: Could not parse intent",
            "messages": [AIMessage(content=f"could not understand translation request. specify both languages and text clearly. Error: {e}")]
        }
    
#placeholders for step 6, unused as of now
def generate_rag_query(state: TranslationState) -> dict: return {} 
def retrieve_context(state: TranslationState) -> dict: return {}
def tool_router(state: TranslationState) -> str: return END
def direct_translate(state: TranslationState) -> dict: return {}

def build_graph():
    workflow = StateGraph(TranslationState)
    
    workflow.add_node("identify_intent", identify_intent)
    
    workflow.add_edge(START, "identify_intent")
    #placeholder nodes with no functionality yet, but needed to illustrate the graph process
    workflow.add_node("generate_rag_query", generate_rag_query)
    workflow.add_node("retrieve_context", retrieve_context)
    workflow.add_node("tool_router", tool_router)
    workflow.add_node("direct_translate", direct_translate)
    #for now, ends after the first step (testing). will update in later steps
    workflow.add_edge("identify_intent", END) 
    return workflow.compile()

In [21]:
if __name__ == "__main__":
    test_input = HumanMessage(content="नमस्ते, मैं इसे ओड़िया में अनुवाद करना चाहता हूँ: मेरा नाम आकाश है और मुझे वनीला आइसक्रीम खाना बहुत पसंद है।'")
    
    initial_state = {"messages": [test_input]}
    
    graph = build_graph()

    result = graph.invoke(initial_state)

    print("\nResulting State:")
    print(f"Source Language: {result.get('source_lang')}")
    print(f"Target Language: {result.get('target_lang')}")
    print(f"Translation Query: {result.get('query_for_retrieval')}")
    
    # expected output:
    # Source Language: hi (Hindi, implied)
    # Target Language: or (Odia/Oriya, explicit)
    # Translation Query: मेरा नाम आकाश है और मुझे वनीला आइसक्रीम खाना बहुत पसंद है।




Resulting State:
Source Language: hi
Target Language: or
Translation Query: मेरा नाम आकाश है और मुझे वनीला आइसक्रीम खाना बहुत पसंद है।
